# The NVSHMEM Library

## Introduction

Many scalable applications perform fine-grained communication that’s tightly coupled with computation. For such applications, existing communication libraries like MPI that support communication only at kernel boundaries can incur significant kernel launch. They place communication latencies on the critical path to performance. NVSHMEM is a communication library based on OpenSHMEM and designed specifically for clusters of NVIDIA GPUs. NVSHMEM can significantly reduce communication and coordination overheads by allowing programmers to perform these operations from within CUDA kernels and on CUDA streams. 

![nvshmem_memory_model](../../images/nvshmem_memory_model.png)

NVSHMEM provides a partitioned global address space (PGAS) that combines the memory of multiple GPUs into a shared address space that is accessed through the NVSHMEM API. The NVSHMEM API allows for fine-grained, remote data access that takes advantage of the massive parallelism in the GPU to hide communication overheads. By performing communication from within CUDA kernels, NVSHMEM allows us to write long running kernels that reduce the amount of overhead generated by synchronization with the CPU. As a result, NVSHMEM can significantly improve parallel efficiency.

**Note:** The NVSHMEM library requires GPUDirect P2P (for intra-node transfers) and GPUDirect RDMA (for inter-node transfers) technologies to be present.

## NVSHMEM compared with CUDA-aware MPI

The OpenSHMEM standard offers a partitioned global address space (PGAS) library for one-sided communication among CPU memories. NVSHMEM extends the standard OpenSHMEM APIs with support for communication from GPU threads, blocks, and warps;  direct data movement between GPU memories; and deferred execution APIs from the CPU that support CUDA Streams.

![nvshmem_mpi_comparison](../../images/nvshmem_mpi_comparison.png)

As shown above, MPI separates out computation and communication into different phases. Even though the data can be transferred directly from GPU memory as described earlier, the send and receive actions are performed on the CPU and serialization with kernel-based computation is necessary. \

* MPI implementations can incur high locking (or atomics) overheads for shared data structures that are involved in messaging.
* Serialization overheads that result from the MPI message ordering requirements.
* Protocol overheads that result from messages arriving at the receiver before they have posted the corresponding receive operation.

Current state-of-the-art applications that run on GPU clusters typically offload computation phases onto the GPU and rely on the CPU to manage communication between cluster nodes, by using MPI. Depending on the CPU for communication limits strong scalability because of the overhead of repeated kernel launches, CPU-GPU synchronization, underutilization of the GPU during communication phases, and underutilization of the network during compute phases. Some of these issues can be addressed by restructuring the application code to overlap independent compute and communication phases using CUDA streams. These optimizations can lead to complex application code and the benefits usually diminish as the problem size per GPU becomes smaller.

## GPU-initiated communication

NVSHMEM's memory model allows both computation and communication to occur from within the device kernel in the GPU. The benefits of this approach are as follows:

* It eliminates offload latencies to the CPU.
* It enables compute and communication overlap by hiding communication latency behind massive GPU parallelism.
* It makes it easier to express algorithms with inline communication.

NVSHMEM provides get and put APIs, which copy data from and to symmetric objects, respectively. Bulk transfer, scalar transfer, and interleaved versions of these APIs are provided. In addition, Atomic Memory Operations (AMOs) are also provided and can be used to perform atomic updates to symmetric variables. With these APIs, NVSHMEM provides fine-grained and low-overhead access to data that is stored in the PGAS from CUDA kernels. By performing communication from within the kernel, NVSHMEM also allows applications to benefit from the intrinsic latency-hiding capabilities of the GPU warp scheduling hardware.

NVSHMEM also allows any two CUDA threads in a job to synchronize on locations in global memory by using the OpenSHMEM point-to-point synchronization API `nvshmem_wait_until` or collective synchronization APIs like `nvshmem_barrier`.

### Communication Model

NVSHMEM provides get and put APIs, which copy data from and to symmetric objects, respectively. Bulk transfer, scalar transfer, and interleaved versions of these APIs are provided. In addition, Atomic Memory Operations (AMOs) are also provided and can be used to perform atomic updates to symmetric variables. With these APIs, NVSHMEM provides fine-grained and low-overhead access to data that is stored in the PGAS from CUDA kernels. By performing communication from within the kernel, NVSHMEM also allows applications to benefit from the intrinsic latency-hiding capabilities of the GPU warp scheduling hardware.

## Memory Model

An NVSHMEM program consists of data objects that are private to each PE and data objects that are remotely accessible by all PEs. Private data objects are stored in the local memory of each PE and can only be accessed by the PE itself; these data objects cannot be accessed by other PEs via NVSHMEM routines. Private data objects follow the memory model of C. Remotely accessible objects, however, can be accessed by remote PEs using NVSHMEM routines. Remotely accessible data objects are called Symmetric Data Objects. Each symmetric data object has a corresponding object with the same name, type, and size on all PEs where that object is accessible via the NVSHMEM API.

NVSHMEM dynamic memory allocation routines (e.g., nvshmem_malloc) allow collective allocation of Symmetric Data Objects on a special memory region called the Symmetric Heap. The Symmetric Heap is created during the execution of a program at a memory location determined by the NVSHMEM library. The Symmetric Heap may reside in different memory regions on different PEs. 

### Pointers to Symmetric Objects

Symmetric data objects are referenced in NVSHMEM operations through the local pointer to the desired remotely accessible object. The address contained in this pointer is referred to as a symmetric address. Every symmetric address is also a local address that is valid for direct memory access; however, not all local addresses are symmetric. Manipulation of symmetric addresses passed to NVSHMEM routines—including pointer arithmetic, array indexing, and access of structure or union members—are permitted as long as the resulting local pointer remains within the same symmetric allocation or object. Symmetric addresses are only valid at the PE where they were generated.

### Visibility Guarantees

On systems with both NVLink and InfiniBand, the NVSHMEM synchronization operations including nvshmem_barrier, nvshmem_barrier_all, nvshmem_quiet, nvshmem_wait_until_*, and nvshmem_test_* only guarantee visibility of updates to the local PE’s symmetric objects. However, on systems with only NVLink, these operations guarantee global visibility of updates to symmetric objects.

## Using NVSHMEM

An NVSHMEM job represents a single program, multiple data (SPMD) parallel execution. Each PE is assigned an integer identifier (ID), that ranges from zero to one less than the total number of PEs. PE IDs are used to identify the source or destination process in OpenSHMEM operations and are also used by application developers to assign work to specific processes in an NVSHMEM job.

All PEs in an NVSHMEM job must simultaneously, i.e. collectively, call the NVSHMEM initialization routine before an NVSHMEM operation can be performed. Similarly, before exiting, PEs must also collectively call the NVSHMEM finalization function. After initialization, a PE’s ID and the total number of running PEs can be queried. PEs communicate and share data through symmetric memory that is allocated from a symmetric heap that is located in GPU memory. This memory is allocated by using the CPU-side NVSHMEM allocation API. Memory that is allocated by using any other method is considered private to the allocating PE and is not accessible by other PEs.

### MPI and NVSHMEM

NVSHMEM can be used with MPI. The following code snippet shows how NVSHMEM can be initialized in an MPI program. In this program, we assume that each MPI process is also an NVSHMEM PE, where each process has both an MPI rank and an NVSHMEM rank.

```c
int main(int argc, char *argv[]) {
    int rank, ndevices;

    nvshmemx_init_attr_t attr;
    MPI_Comm comm = MPI_COMM_WORLD;
    attr.mpi_comm = &comm;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    cudaGetDeviceCount(&ndevices);
    cudaSetDevice(rank % ndevices);
    nvshmemx_init_attr(NVSHMEMX_INIT_WITH_MPI_COMM, &attr);

    // ...

    nvshmem_finalize();
    MPI_Finalize();
    return 0;
}
```

As shown in this example, the MPI library should be initialized first. After MPI is initialized, the MPI rank can be queried and used to set the CUDA device. An `nvshmemx_init_attr_t` structure is created and the `mpi_comm` field is assigned a reference to an MPI communicator handle. To enable MPI compatibility mode, the `nvshmemx_init_attr` operation is used instead of `nvshmem_init`. 

### Hello World Example: Left Shift Kernel with NVSHMEM and MPI

The code snippet below shows a simple example of NVSHMEM usage within a CUDA kernel where PEs form a communication ring.

```c
__global__ void simple_shift(int *destination) {
    int mype = nvshmem_my_pe();
    int npes = nvshmem_n_pes();
    int peer = (mype + 1) % npes;

    nvshmem_int_p(destination, mype, peer);
}

int main (int argc, char *argv[]) {
    int mype_node, msg;
    cudaStream_t stream;
    int rank, nranks;
    MPI_Comm mpi_comm = MPI_COMM_WORLD;
    nvshmemx_init_attr_t attr;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &nranks);

    attr.mpi_comm = &mpi_comm;
    nvshmemx_init_attr(NVSHMEMX_INIT_WITH_MPI_COMM, &attr);
    mype_node = nvshmem_team_my_pe(NVSHMEMX_TEAM_NODE);

    cudaSetDevice(mype_node);
    cudaStreamCreate(&stream);
    int *destination = (int *) nvshmem_malloc (sizeof(int));

    simple_shift<<<1, 1, 0, stream>>>(destination);
    nvshmemx_barrier_all_on_stream(stream);
    cudaMemcpyAsync(&msg, destination, sizeof(int), cudaMemcpyDeviceToHost, stream);

    cudaStreamSynchronize(stream);
    printf("%d: received message %d\n", nvshmem_my_pe(), msg);

    nvshmem_free(destination);
    nvshmem_finalize();
    MPI_Finalize();
    return 0;
}
```

This example begins in main by initializing the NVSHMEM and MPI libraries, querying the PE’s ID in the on-node team, and using the on-node ID to set the CUDA device. The device must be set before you allocate memory or launch a kernel. A stream is created and a symmetric integer called `destination` is allocated on every PE. Finally, the `simple_shift` kernel is launched on one thread with a pointer to this symmetric object as its argument.

![nvshmem_left_shift_output](../../images/nvshmem_left_shift_output.png)

This kernel queries the global PE ID and the number of executing PEs. It then performs a single-element integer put operation to write the calling PE’s ID into destination at the PE with the next highest ID, or in the case of the PE with the highest ID, 0. The kernel is launched asynchronously on stream, followed by an NVSHMEM barrier on the stream to ensure that all updates have completed, and an asynchronous copy to copy the updated destination value to the host. The stream is synchronized and the result is printed. 

Finally, the destination buffer is freed and the NVSHMEM library is finalized before the program exits.

#### Compilation

Compiling NVSHMEM programs requires that the `NVSHMEM_HOME` environment variable is set. The canonical compilation command looks like this:

```bash
nvcc -rdc=true -ccbin=mpic++ -gencode=$NVCC_GENCODE -I $NVSHMEM_HOME/include left_shift.cu \
     -o left_shift -L $NVSHMEM_HOME/lib -lnvshmem_host -lnvshmem_device -lcuda
```

---

## Running this lab on a single Perlmutter GPU node

As in the NCCL lab, this notebook has been adapted from material written for a DGX cluster with 8 GPUs per node and an OpenMPI/`mpirun` launcher. **We are running on one Perlmutter GPU node with 4 NVIDIA A100-SXM4-40GB GPUs.** NVSHMEM needs more care than NCCL did, so read this section before compiling.

### Why this lab is the fussiest of the series

NVSHMEM has to agree with three separate things on this machine: the CUDA toolkit, the MPI implementation, and the network transport. The NCCL lab only had to agree with the first two.

* **GPUDirect P2P is required** for intra-node transfers. All 4 A100s on a Perlmutter node are NVLink-connected, so this is satisfied.
* **GPUDirect RDMA** is only needed for inter-node transfers, which we are not doing. `NVSHMEM_REMOTE_TRANSPORT=none` tells NVSHMEM to skip network setup entirely, which removes a whole class of startup failures.

### Launching

Identical to the NCCL lab:

```bash
srun --overlap --mpi=cray_shasta -N 1 -n <ranks> -c 32 --gpus-per-task=1 --gpu-bind=none ./jacobi_nvshmem ...
```

`--gpu-bind=none` remains mandatory — the program calls `cudaSetDevice()` on its node-local PE ID and needs to see all 4 GPUs. Keep `--mpi=cray_shasta` explicit on every launch: if you load NERSC's `nvshmem` module it sets `SLURM_MPI_TYPE=pmix`, and PMIx makes Cray MPICH initialise each rank as its own singleton `MPI_COMM_WORLD` — which shows up as the whole run printing once per rank with `Num GPUs: 1.`

Start by setting up the environment.

In [1]:
# --- Environment for the NVSHMEM lab on Perlmutter -------------------------------
# HPC SDK 26.5's NVSHMEM 3.6.5, matching the CUDA 13.2 toolchain already in PATH.
%env NVSHMEM_HOME=/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/comm_libs/13.2/nvshmem
%env CUDA_HOME=/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2

# MPI is used only for the host-side L2 norm Allreduce and the unique-ID
# broadcast, so Cray MPICH does not need its GPU Transport Layer.
%env MPICH_GPU_SUPPORT_ENABLED=0

# Single node: skip all network transport setup.
%env NVSHMEM_REMOTE_TRANSPORT=none

# NERSC sets this for their own NVSHMEM module; harmless here and occasionally necessary.
%env NVSHMEM_DISABLE_CUDA_VMM=1

# Enable NVSHMEM's NVTX instrumentation so the profiling section has something to show.
%env NVSHMEM_NVTX=common

env: NVSHMEM_HOME=/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/comm_libs/13.2/nvshmem
env: CUDA_HOME=/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2
env: MPICH_GPU_SUPPORT_ENABLED=0
env: NVSHMEM_REMOTE_TRANSPORT=none
env: NVSHMEM_DISABLE_CUDA_VMM=1
env: NVSHMEM_NVTX=common


In [2]:
# Cray MPICH - the Makefile aborts if MPI_HOME is unset.
import os
os.environ["MPI_HOME"] = os.environ.get("CRAY_MPICH_DIR", "")
print("MPI_HOME =", os.environ["MPI_HOME"] or "*** EMPTY - is cray-mpich loaded? ***")

MPI_HOME = /opt/cray/pe/mpich/9.1.0/ofi/gnu/12.3


Now compile the Left Shift "hello world":

In [3]:
!cd ../../source_code/nvshmem/ && make clean && make left_shift

rm -rf jacobi_nvshmem left_shift *.o *.qdrep *.sqlite *.nsys-rep
nvcc -dc -Xcompiler -fopenmp -lineinfo -gencode arch=compute_80,code=sm_80 -gencode arch=compute_80,code=compute_80 -std=c++17 -I/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/comm_libs/13.2/nvshmem/include -I/opt/cray/pe/mpich/9.1.0/ofi/gnu/12.3/include -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/targets/x86_64-linux/lib/stubs/ -lnvidia-ml -I/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/include left_shift.cu -c -o left_shift.o
nvcc -gencode arch=compute_80,code=sm_80 -gencode arch=compute_80,code=compute_80 -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/targets/x86_64-linux/lib/stubs/ -lnvidia-ml -I/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/include left_shift.o -o left_shift -ccbin=mpic++ -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/comm_libs/13.2/nvshmem/lib -lnvshmem_host -lnvshmem_device -L/opt/cray/pe/mpich/9.1.0/ofi/gnu/12.3/lib -lmpi -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/lib64 -lcuda -lcudart -Xlinker

Now run the Left Shift program. Each PE writes its own ID into its right-hand neighbour's `destination`, wrapping around, so PE *n* should receive the message from PE *n-1* (and PE 0 from the highest PE).

Start with 2 GPUs, then use all 4.

In [5]:
# 2 PEs
!cd ../../source_code/nvshmem/ && srun --overlap --mpi=cray_shasta -N 1 -n 2 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./left_shift

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 2 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
0: received message 1
1: received message 0


In [6]:
# 4 PEs - all GPUs on the node
!cd ../../source_code/nvshmem/ && srun --overlap --mpi=cray_shasta -N 1 -n 4 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./left_shift

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 4 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
1: received message 0
2: received message 1
0: received message 3
3: received message 2


Expected output with 2 GPUs (order of lines will vary):

```bash
0: received message 1
1: received message 0
```

Expected output with 4 GPUs:

```bash
0: received message 3
1: received message 0
2: received message 1
3: received message 2
```

If this hangs or aborts inside `nvshmemx_init_attr`, the UID edits above have most likely not been applied — the kernel itself is three lines, so it is almost never your code. Re-run with `NVSHMEM_DEBUG=INFO` and check which bootstrap was selected: you want `UID`, not `MPI`.

Now, let us learn more about NVSHMEM APIs.

### Heap Memory Allocation

```c
void *nvshmem_malloc(size_t size)
```

The `nvshmem_malloc` routine returns the symmetric address of a block of at least `size` bytes, which shall be suitably aligned so that it may be assigned to a pointer to any type of object. This space is allocated from the symmetric heap (in contrast to `malloc`, which allocates from the private heap). 

### Thread-level communication

To allow fine-grained thread-level communication and computation overlap, NVSHMEM provides thread-level communication APIs. 

![nvshmem_thread_level_comm](../../images/nvshmem_thread_level_comm.png)

The device kernel for a stencil-based program, like our Jacobi solver, looks as follows:

```c
__global__ void stencil_single_step(float *u, float *v, …) {
    int ix = get_ix(blockIdx, blockDim, threadIdx);
    int iy = get_iy(blockIdx, blockDim, threadIdx);
    compute(u, v, ix, iy);
    // Thread-level data communication API
    if (iy == 1)
        nvshmem_float_p(u+(ny+1)*nx+ix, u[nx+ix], top_pe);
    if (iy == ny)
        nvshmem_float_p(u+ix, u[ny*nx+ix], bottom_pe);
}
```

To ensure all communication is complete at the end of each iteration, use `nvshmem_barrier_all_on_stream` function in the host-side loop as follows:

```c
for (int iter = 0; iter < N; iter++) {
    swap(u, v);
    stencil_single_step<<<..., stream>>>(u, v, …);
    nvshmem_barrier_all_on_stream(stream);
}
```

### Thread-group level communication

NVSHMEM operations can be issued by all threads in a block/warp and this technique is more efficient for data transfers over networks like IB. It allows overlap at inter-warp or inter-block level.

Here's an example for block-level communication:

```c
// Thread block-level communication API
int boffset = get_block_offset(blockIdx,blockDim);
if (blockIdx.y == 0)
    nvshmemx_float_put_nbi_block(u+(ny+1)*nx+boffset, u+nx+boffset, blockDim.x, top_pe);
if (blockIdx.y == (blockDim.y-1))
    nvshmemx_float_put_nbi_block(u+boffset, u+ny*nx+boffset, blockDim.x, bottom_pe);
```

### Program-level Barrier

The `nvshmem_barrier_all` routine is a mechanism for synchronizing all PEs at once. This routine blocks the calling PE until all PEs have called `nvshmem_barrier_all`. In a multithreaded NVSHMEM program, only the calling thread is blocked, however, it may not be called concurrently by multiple threads in the same PE.

## Implementation Exercise

Open the [jacobi_nvshmem.cu](../../source_code/nvshmem/jacobi_nvshmem.cu) file. Alternatively, you can navigate to `CFD/English/C/source_code/nvshmem/` directory in Jupyter's file browser in the left pane. Then, click to open the `jacobi_nvshmem.cu` file.

Observe the following steps that occur prior to the Jacobi solver's iterative loop:

1. MPI is initialized, local rank and local size (number of GPUs per node) is determined, and current GPU is set. 
2. The default value of NVSHMEM's symmetric heap size is 1GB, which is insufficient for large grid sizes. This size is reset by setting the `NVSHMEM_SYMMETRIC_SIZE` environment variable from within the program.
3. NVSHMEM is initialized.
4. Current PE ID is stored in `mype` variable.
5. `nvshmem_malloc` allocates chunks of equal size across all PEs.

Now, NVSHMEM APIs like barriers work naturally with CUDA streams. Thus, we employ a small optimization called "delayed norm reset". We reset the norm in a separate `reset_l2_norm_stream` and perform norm device kernel computation, and device-local norm copy back to host on `compute_stream`.

We use `reset_l2_norm_done[curr]`event to ensure that norm reset is performed before we launch `jacobi_kernel` as we make the `compute_stream` wait on it. Moreover, we record the copy of the norm back to host in `l2_norm_bufs[curr].copy_done` event and synchronize this event before calling `MPI_AllReduce`. Finally, we reset everything before moving on to the next iteration and record the norm copy event in `reset_l2_norm_done[prev]`.

This ensures that the CPU-centric L2 norm reduction is kept separate in the GPU (using streams) from the GPU-centric Jacobi solver computations and GPU-initiated communications (within the device kernel).

After understanding the program flow, implement the following marked as `TODO`:

* In the `jacobi_kernel` device kernel:
  - Use block-level NVSHMEM put communication API to transfer the halo pointed to by `a_new + iy_start * nx + block_ix` to `a_new + top_iy * nx + block_ix` in the `top_pe`.
  - Similarily, transfer the halo pointed to by `a_new + (iy_end - 1) * nx + block_ix` to `a_new + bottom_iy * nx + block_ix` in the `bottom_pe`.
* In the iterative Jacobi (`while`) loop:
  - Use `cudaStreamWaitEvent` on `compute_stream` to wait for `reset_l2_norm_done` event to complete for `curr` iteration.
  - Put a barrier at the "compute_stream" stream level before performing L2 norming calculation.
  - Record the event `l2_norm_bufs[curr].copy_done` for `compute_stream`.
  - Record the L2 norm reset in `reset_l2_norm_done[prev]` for `reset_l2_norm_stream`. Understand why we use `prev` iteration: the stream waits for this event in the next iteration with `curr` iteration value.

After implementing these, compile the Jacobi solver:

In [7]:
!cd ../../source_code/nvshmem/ && make jacobi_nvshmem

nvcc -dc -Xcompiler -fopenmp -lineinfo -gencode arch=compute_80,code=sm_80 -gencode arch=compute_80,code=compute_80 -std=c++17 -I/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/comm_libs/13.2/nvshmem/include -I/opt/cray/pe/mpich/9.1.0/ofi/gnu/12.3/include -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/targets/x86_64-linux/lib/stubs/ -lnvidia-ml -I/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/include jacobi_nvshmem.cu -c -o jacobi_nvshmem.o
nvcc -gencode arch=compute_80,code=sm_80 -gencode arch=compute_80,code=compute_80 -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/targets/x86_64-linux/lib/stubs/ -lnvidia-ml -I/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/include jacobi_nvshmem.o -o jacobi_nvshmem -ccbin=mpic++ -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/comm_libs/13.2/nvshmem/lib -lnvshmem_host -lnvshmem_device -L/opt/cray/pe/mpich/9.1.0/ofi/gnu/12.3/lib -lmpi -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/lib64 -lcuda -lcudart -Xlinker -rpath -Xlinker /opt/nvidia/hpc_sdk/Linux_x86_64

Ensure that there are no errors. Now validate your implementation.

As in the NCCL lab we scale the grid with the GPU count so per-GPU work stays constant: $2K\times4K$ on 2 GPUs, $4K\times8K$ on 4.

A correct run prints `Num GPUs: 2.` (or `4.`) **once**. Output repeated once per rank with `Num GPUs: 1.` means the MPI bootstrap failed — check `--mpi=cray_shasta`.

In [8]:
# 2 GPUs
!cd ../../source_code/nvshmem/ && srun --overlap --mpi=cray_shasta -N 1 -n 2 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./jacobi_nvshmem -nx 2048 -ny 4096

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 2 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
Setting environment variable NVSHMEM_SYMMETRIC_SIZE = 36927897
Single GPU jacobi relaxation: 1000 iterations on 2048 x 4096 mesh
  100, 0.452182
  200, 0.268726
  300, 0.198221
  400, 0.159730
  500, 0.135100
  600, 0.117824
  700, 0.104951
  800, 0.094942
  900, 0.086909
 1000, 0.080300
Jacobi relaxation: 1000 iterations on 2048 x 4096 mesh
  100, 0.455608
  200, 0.269740
  300, 0.198718
  400, 0.160030
  500, 0.135304
  600, 0.117971
  700, 0.105064
  800, 0.095031
  900, 0.086981
 1000, 0.080360
Num GPUs: 2.
4096x2048: 1 GPU:   0.1214 s, 2 GPUs:   0.0874 s, speedup:     1.39, efficiency:    69.49 


In [9]:
# 4 GPUs
!cd ../../source_code/nvshmem/ && srun --overlap --mpi=cray_shasta -N 1 -n 4 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./jacobi_nvshmem -nx 4096 -ny 8192

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 4 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
Setting environment variable NVSHMEM_SYMMETRIC_SIZE = 73891840
Single GPU jacobi relaxation: 1000 iterations on 4096 x 8192 mesh
  100, 0.639626
  200, 0.380139
  300, 0.280415
  400, 0.225973
  500, 0.191138
  600, 0.166702
  700, 0.148496
  800, 0.134341
  900, 0.122979
 1000, 0.113633
Jacobi relaxation: 1000 iterations on 4096 x 8192 mesh
  100, 0.644473
  200, 0.381573
  300, 0.281119
  400, 0.226398
  500, 0.191425
  600, 0.166911
  700, 0.148656
  800, 0.134467
  900, 0.123082
 1000, 0.113718
Num GPUs: 4.
8192x4096: 1 GPU:   0.4606 s, 4 GPUs:   0.1470 s, speedup:     3.13, efficiency:    78.35 


### Interpreting what you just saw

Unlike NCCL, NVSHMEM should look respectable even at 1000 iterations. The library is built for low latency and does not carry the same heavyweight per-call setup, so there is much less fixed overhead to amortise. If your NVSHMEM efficiency at 1000 iterations is already close to what NCCL reached at 5000, that is the expected result and is the main point of this lab.

Run the longer version anyway, both to confirm that claim and to have a like-for-like comparison against the numbers you recorded in the NCCL lab.

In [10]:
# 2 GPUs, 5000 iterations
!cd ../../source_code/nvshmem/ && srun --overlap --mpi=cray_shasta -N 1 -n 2 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./jacobi_nvshmem -nx 2048 -ny 4096 -niter 5000

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 2 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
Setting environment variable NVSHMEM_SYMMETRIC_SIZE = 36927897
Single GPU jacobi relaxation: 5000 iterations on 2048 x 4096 mesh
  100, 0.452182
  200, 0.268726
  300, 0.198221
  400, 0.159730
  500, 0.135100
  600, 0.117824
  700, 0.104951
  800, 0.094942
  900, 0.086909
 1000, 0.080300
 1100, 0.074755
 1200, 0.070028
 1300, 0.065944
 1400, 0.062374
 1500, 0.059225
 1600, 0.056423
 1700, 0.053912
 1800, 0.051646
 1900, 0.049591
 2000, 0.047716
 2100, 0.045999
 2200, 0.044419
 2300, 0.042960
 2400, 0.041608
 2500, 0.040351
 2600, 0.039179
 2700, 0.038083
 2800, 0.037056
 2900, 0.036091
 3000, 0.035183
 3100, 0.034326
 3200, 0.033517
 3300, 0.032750
 3400, 0.032023
 3500, 0.031332
 3600, 0.030676
 3700, 0.030050
 3800, 0.029453
 3900, 0.028883
 4000, 0.028338
 4100, 0.027817
 4200, 0.027317
 4300, 0.026837
 4400, 0.0263

In [11]:
# 4 GPUs, 5000 iterations
!cd ../../source_code/nvshmem/ && srun --overlap --mpi=cray_shasta -N 1 -n 4 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./jacobi_nvshmem -nx 4096 -ny 8192 -niter 5000

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 4 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
Setting environment variable NVSHMEM_SYMMETRIC_SIZE = 73891840
Single GPU jacobi relaxation: 5000 iterations on 4096 x 8192 mesh
  100, 0.639626
  200, 0.380139
  300, 0.280415
  400, 0.225973
  500, 0.191138
  600, 0.166702
  700, 0.148496
  800, 0.134341
  900, 0.122979
 1000, 0.113633
 1100, 0.105790
 1200, 0.099105
 1300, 0.093329
 1400, 0.088281
 1500, 0.083828
 1600, 0.079865
 1700, 0.076314
 1800, 0.073110
 1900, 0.070204
 2000, 0.067553
 2100, 0.065125
 2200, 0.062891
 2300, 0.060827
 2400, 0.058916
 2500, 0.057138
 2600, 0.055481
 2700, 0.053932
 2800, 0.052480
 2900, 0.051116
 3000, 0.049832
 3100, 0.048620
 3200, 0.047476
 3300, 0.046392
 3400, 0.045364
 3500, 0.044388
 3600, 0.043459
 3700, 0.042574
 3800, 0.041731
 3900, 0.040925
 4000, 0.040154
 4100, 0.039417
 4200, 0.038711
 4300, 0.038033
 4400, 0.0373

Compare the 1000- and 5000-iteration figures. The gap between them should be much smaller than it was for NCCL, because NVSHMEM has little fixed start-up cost to spread out.

Then compare the 5000-iteration NVSHMEM numbers against your NCCL and CUDA-aware MPI results for the same GPU count and grid on this same node. That three-way comparison is the payoff of the whole bootcamp.

## Profiling

NVSHMEM's NVTX instrumentation is enabled by `NVSHMEM_NVTX=common`, which was set in the environment cell at the top.

Two Perlmutter-specific changes from the original notebook:

* **`nsys` goes inside `srun`,** not around it, so each rank is profiled in its own process. We get one report per PE, named by `SLURM_PROCID`.
* **Reports are written to `$SCRATCH`.** NERSC global homes is GPFS, and the Nsight Systems report writer uses file operations GPFS does not support — it fails at the import stage with `errno 524` after collection has already succeeded. `$SCRATCH` is Lustre and works.

We also pass `--mpi-impl=mpich` so `nsys` does not have to guess which MPI to trace, and drop the `--nic-metrics` and `ucx` options from the original, which assume InfiniBand and UCX respectively.

In [ ]:
!cd ../../source_code/nvshmem/ && export OUT=$SCRATCH/nvshmem_prof && mkdir -p $OUT && \
 srun --overlap --mpi=cray_shasta -N 1 -n 4 -c 32 --gpus-per-task=1 --gpu-bind=none \
 nsys profile --trace=cuda,nvtx,mpi --mpi-impl=mpich --stats=true --force-overwrite true \
 -o $OUT/jacobi_nvshmem_%q{SLURM_PROCID} \
 ./jacobi_nvshmem -nx 4096 -ny 8192 -niter 5000 -skip_single_gpu

In [ ]:
!ls -lh $SCRATCH/nvshmem_prof/
# Copy rank 0 back so it is downloadable from Jupyter's file browser.
!mkdir -p ../../source_code/nvshmem/output_profiler && \
 cp $SCRATCH/nvshmem_prof/jacobi_nvshmem_0.nsys-rep ../../source_code/nvshmem/output_profiler/

Download `output_profiler/jacobi_nvshmem_0.nsys-rep` (hold <mark>Shift</mark> and <mark>Right-Click</mark> in Jupyter's file browser, then "Save Link As") and open it in the Nsight Systems GUI.

### What to look for in the timeline

Most NVSHMEM calls appear in the **NVTX Push-Pop** summary in the cell output above — a contrast with NCCL, where the asynchronous device-side execution made that table nearly useless. NVSHMEM is visible both on the **CUDA Hardware** row and on the **Threads** row.

Zoom into a few consecutive iterations and measure:

1. The **total time between two Jacobi kernel launches**.
2. The latency of the **`barrier_on_stream`** wrapper.

The difference between them is the per-PE idle time. Work it out for your own run and set it beside the equivalent numbers from the NCCL and CUDA-aware MPI labs.

Then look for the structural result that makes NVSHMEM different: **`MPI_Allreduce` and the device-to-host copy of the L2 norm (shown in green) overlap with the Jacobi kernel itself.** The halo exchange is happening *inside* the kernel via `nvshmemx_float_put_nbi_block`, so there is no separate communication phase to wait on. That concurrency is what the "delayed norm reset" stream structure in the source code exists to enable, and it is the reason NVSHMEM keeps GPUs busier than a communicate-then-compute model can.

![nvshmem_profiler_report](../../images/nvshmem_profiler_report.png)

*(The screenshot above is from the original material, captured on different hardware with a larger grid. Use it to recognise what the timeline looks like, not as a comparison for your own numbers.)*

### Troubleshooting

If a run fails or hangs, these are the usual causes on this system:

* **Hang or abort during init** - check the UID edits were applied. `NVSHMEM_DEBUG=INFO` prints the bootstrap NVSHMEM chose; `MPI` there means the edits are missing, and a `libmpi.so.40: cannot open shared object file` message is the OpenMPI/Cray MPICH mismatch described at the top of this notebook.
* **`nvshmem_malloc` fails for larger grids** - the symmetric heap default is 1 GB. The source sets `NVSHMEM_SYMMETRIC_SIZE` internally, but you can override it in the environment if you push the grid size up.
* **Errors mentioning CUDA VMM** - `NVSHMEM_DISABLE_CUDA_VMM=1` is already set in the environment cell above; confirm it took effect in this kernel.
* **Network/IB errors on a single-node run** - confirm `NVSHMEM_REMOTE_TRANSPORT=none` is actually set in this kernel's environment.

We now have an in-depth view of programming with NVSHMEM to achieve optimal performance by using GPU-initiated asynchronous communication and minimal synchronization calls. This concludes the overview of various multi-node multi-GPU programming models and approaches in this bootcamp.

Click link to the home notebook below through which all other notebooks are accessible:

# [HOME](../../../start_here.ipynb)

---
## Links and Resources

* [Concepts: Accelerating IO in data-centers](https://developer.nvidia.com/blog/accelerating-io-in-the-modern-data-center-network-io/)
* [Programming Concepts: NVSHMEM User Guide](https://docs.nvidia.com/hpc-sdk/nvshmem/api/docs/introduction.html)
* [Programming Concepts: Scaling Scientific Computing with NVSHMEM](https://developer.nvidia.com/blog/scaling-scientific-computing-with-nvshmem/)
* [Documentation: NCCL API](https://docs.nvidia.com/hpc-sdk/nvshmem/api/docs/api.html)
* [Code: Multi-GPU Programming Models](https://github.com/NVIDIA/multi-gpu-programming-models)
* [Code: GPU Bootcamp](https://github.com/gpuhackathons-org/gpubootcamp/)

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

## Licensing
Copyright © 2022 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
